In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [1]:
# Pin a known-compatible HF stack (peft/transformers/torch)
%pip install -q --upgrade --force-reinstall --no-cache-dir "torch==2.5.1" "transformers==4.45.2" "accelerate==0.34.2" "datasets==2.21.0" "peft==0.12.0" "bitsandbytes==0.43.3" "trl==0.10.1" "huggingface_hub>=0.24,<1.0" "wandb"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 265.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 273.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 273.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 242.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 270.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 304.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 261.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 182.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 187.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 370.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 284.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 305.1 MB/s eta 0:00:00
   ━━━━━━━━━

In [2]:
import os

# Force Transformers to use PyTorch-only path in mixed Kaggle environments.
os.environ.setdefault("USE_TORCH", "1")
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("TRANSFORMERS_NO_FLAX", "1")

# Reduce CUDA memory fragmentation before torch is imported.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Try environment variable first; login happens in the unified imports cell.
hf_token = os.environ.get("HF_TOKEN")

# Kaggle fallback: read from Secrets if env var is not injected.
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        hf_token = user_secrets.get_secret("HF_TOKEN")
    except Exception as e:
        print(f"Kaggle secret fetch failed: {e}")

# Runtime-aware paths for Kaggle or Colab.
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB = "COLAB_GPU" in os.environ

if IS_KAGGLE:
    CORPUS_PATH = os.environ.get(
        "HAMROAI_CORPUS_PATH",
        "/kaggle/input/datasets/darksunnp/hamroai-cpt-v1/all_cleaned_deduped.jsonl",
    )
    WORK_DIR = "/kaggle/working"
elif IS_COLAB:
    CORPUS_PATH = os.environ.get(
        "HAMROAI_CORPUS_PATH",
        "/content/all_cleaned_deduped.jsonl",
    )
    WORK_DIR = "/content"
else:
    CORPUS_PATH = os.environ.get("HAMROAI_CORPUS_PATH", "./all_cleaned_deduped.jsonl")
    WORK_DIR = "."

CHECKPOINT_DIR = f"{WORK_DIR}/hamroai-cpt-checkpoints"
ADAPTER_DIR = f"{WORK_DIR}/hamroai-nepali-lora"
TOKENIZED_CACHE_DIR = f"{WORK_DIR}/hamroai-tokenized-cache"

# Optional tokenizer controls (default switched to custom tokenizer).
USE_CUSTOM_TOKENIZER = os.environ.get("HAMROAI_USE_CUSTOM_TOKENIZER", "1") == "1"
CUSTOM_TOKENIZER_PATH = os.environ.get(
    "HAMROAI_CUSTOM_TOKENIZER_PATH",
    "/kaggle/input/datasets/darksunnp/hamroai-cpt-v1/nepali_bpe_32k.json",
)

# Throughput-oriented defaults (override via env vars if needed).
MAX_LENGTH = int(os.environ.get("HAMROAI_MAX_LENGTH", "512"))
NUM_PROC = max(1, min(4, os.cpu_count() or 2))
TRAIN_DOC_LIMIT = int(os.environ.get("HAMROAI_TRAIN_DOC_LIMIT", "350000"))
VAL_DOC_LIMIT = int(os.environ.get("HAMROAI_VAL_DOC_LIMIT", "4500"))
MAX_STEPS = int(os.environ.get("HAMROAI_MAX_STEPS", "2200"))

print(f"Runtime: {'Kaggle' if IS_KAGGLE else ('Colab' if IS_COLAB else 'Local')}")
print(f"Corpus path: {CORPUS_PATH}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"Tokenized cache dir: {TOKENIZED_CACHE_DIR}")
print(f"Use custom tokenizer: {USE_CUSTOM_TOKENIZER}")
print(f"Custom tokenizer path: {CUSTOM_TOKENIZER_PATH}")
print(f"Train doc limit: {TRAIN_DOC_LIMIT}")
print(f"Val doc limit: {VAL_DOC_LIMIT}")
print(f"Max steps: {MAX_STEPS}")
print(f"Max length: {MAX_LENGTH}")

Runtime: Kaggle
Corpus path: /kaggle/input/datasets/darksunnp/hamroai-cpt-v1/all_cleaned_deduped.jsonl
Checkpoint dir: /kaggle/working/hamroai-cpt-checkpoints
Tokenized cache dir: /kaggle/working/hamroai-tokenized-cache
Use custom tokenizer: True
Custom tokenizer path: /kaggle/input/datasets/darksunnp/hamroai-cpt-v1/nepali_bpe_32k.json
Train doc limit: 350000
Val doc limit: 4500
Max steps: 2200
Max length: 512


In [5]:
%pip uninstall -y numpy pandas pyarrow
%pip install -q --no-cache-dir --force-reinstall numpy==1.26.4 pandas==2.2.2 pyarrow==17.0.0

Found existing installation: numpy 2.4.3
Uninstalling numpy-2.4.3:
  Successfully uninstalled numpy-2.4.3
Found existing installation: pandas 3.0.1
Uninstalling pandas-3.0.1:
  Successfully uninstalled pandas-3.0.1
Found existing installation: pyarrow 23.0.1
Uninstalling pyarrow-23.0.1:
  Successfully uninstalled pyarrow-23.0.1
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 295.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 240.7 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 258.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 382.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 400.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.5/348.5 kB 401.2 MB/s eta 0:00:00
ERROR: pip's de

In [3]:
# Unified imports and environment health checks.
import gc
import importlib
import os
import subprocess
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset, load_from_disk
from huggingface_hub import login
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    PreTrainedTokenizerFast,
    Trainer,
    TrainingArguments,
    trainer_utils,
  )

# Login after imports (token fetched in Cell 2).
if hf_token:
    login(token=hf_token)
    print("Hugging Face login successful.")
else:
    print("HF_TOKEN not found in env or Kaggle Secrets. Login skipped.")

modules = ["transformers", "accelerate", "datasets", "peft", "bitsandbytes"]
for m in modules:
    mod = importlib.import_module(m)
    print(f"{m}: {getattr(mod, '__version__', 'unknown')}")

print("numpy:", np.__version__)
print("numpy.char available:", hasattr(np, "char"))
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu count:", torch.cuda.device_count())
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Hugging Face login successful.
transformers: 4.45.2
accelerate: 0.34.2
datasets: 2.21.0
peft: 0.12.0
bitsandbytes: 0.43.3
numpy: 1.26.4
numpy.char available: True
torch: 2.5.1+cu124
cuda available: True
gpu count: 2


In [4]:
# Tokenizer A/B diagnostics: base Qwen tokenizer vs custom Nepali BPE tokenizer.
# This is a cheap check before full training, not a final quality verdict.
samples = [
    "नेपालको राजधानी काठमाडौं हो।",
    "आज मौसम राम्रो छ र म काम गरिरहेछु।",
    "हामी नेपाली भाषाको लागि राम्रो भाषा मोडेल बनाउन चाहन्छौं।",
    "कृत्रिम बुद्धिमत्ता शिक्षा, स्वास्थ्य र शासनमा उपयोगी हुन सक्छ।",
    "नेपालका विभिन्न जिल्लामा फरक फरक भाषिक विविधता पाइन्छ।",
]

base_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B")
if base_tok.pad_token is None:
    base_tok.pad_token = base_tok.eos_token

custom_ok = os.path.exists(CUSTOM_TOKENIZER_PATH)
if custom_ok:
    custom_tok = PreTrainedTokenizerFast(
        tokenizer_file=CUSTOM_TOKENIZER_PATH,
        bos_token="<s>",
        eos_token="</s>",
        unk_token="<unk>",
        pad_token="<pad>",
    )
else:
    custom_tok = None
    print(f"Custom tokenizer not found at: {CUSTOM_TOKENIZER_PATH}")

def avg_tokens(tok, texts):
    lengths = [len(tok(t, add_special_tokens=False)["input_ids"]) for t in texts]
    return sum(lengths) / max(1, len(lengths)), lengths

def unk_rate(tok, texts):
    if tok.unk_token_id is None:
        return 0.0
    total = 0
    unk = 0
    for t in texts:
        ids = tok(t, add_special_tokens=False)["input_ids"]
        total += len(ids)
        unk += sum(1 for x in ids if x == tok.unk_token_id)
    return (unk / max(1, total)) * 100.0

base_avg, base_lengths = avg_tokens(base_tok, samples)
print(f"Base tokenizer avg tokens: {base_avg:.2f}")
print("Base lengths:", base_lengths)
print(f"Base unk rate: {unk_rate(base_tok, samples):.2f}%")

if custom_tok is not None:
    custom_avg, custom_lengths = avg_tokens(custom_tok, samples)
    print(f"Custom tokenizer avg tokens: {custom_avg:.2f}")
    print("Custom lengths:", custom_lengths)
    custom_unk = unk_rate(custom_tok, samples)
    print(f"Custom unk rate: {custom_unk:.2f}%")
    compression = (base_avg - custom_avg) / max(base_avg, 1e-9) * 100.0
    print(f"Token reduction vs base: {compression:.2f}%")

    # Quick sanity check on one decoded example.
    test_ids = custom_tok(samples[0], add_special_tokens=False)["input_ids"]
    test_decoded = custom_tok.decode(test_ids, skip_special_tokens=True)
    print("Custom decode sample:", test_decoded)

    if custom_unk > 1.0:
        print("Warning: high <unk> rate. Compression may be artificial and quality can drop.")
    elif custom_avg <= base_avg * 0.85:
        print("Signal: strong token efficiency gain with healthy coverage; worth A/B training experiment.")
    else:
        print("Signal: small token efficiency gain; likely not worth switching for LoRA-only runs.")

Base tokenizer avg tokens: 45.40
Base lengths: [27, 31, 56, 60, 53]
Base unk rate: 0.00%
Custom tokenizer avg tokens: 9.40
Custom lengths: [5, 10, 10, 13, 9]
Custom unk rate: 0.00%
Token reduction vs base: 79.30%
Custom decode sample: नेपालकोराजधानीकाठमाडौंहो।
Signal: strong token efficiency gain with healthy coverage; worth A/B training experiment.


In [ ]:
# Optional targeted repair cell (run this instead of the long install cell).
# Use when you hit protobuf/numpy-related import issues.
%pip install -q --upgrade --force-reinstall --no-cache-dir "protobuf==4.25.3" "numpy==1.26.4" "pandas==2.2.2" "pyarrow==17.0.0"

import os
os.environ["USE_TORCH"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
print("Targeted repair applied. Restart kernel/session once, then continue from the version-check cell.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 166.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 164.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 162.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 286.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 317.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.5/348.5 kB 330.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>

In [ ]:
# Aggressive runtime repair for mixed Kaggle environments.
# Run only if you still see errors like: No module named 'numpy.char' or TF/protobuf import crashes.
%pip uninstall -y tensorflow tensorflow-cpu tf-keras keras jax jaxlib 2>nul
%pip install -q --upgrade --force-reinstall --no-cache-dir "numpy==1.26.4" "pandas==2.2.2" "pyarrow==17.0.0" "protobuf==4.25.3"

import os, numpy as np
os.environ["USE_TORCH"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
print("Repair complete. numpy:", np.__version__)
print("Restart kernel/session now, then run from the version-check cell.")

Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
Found existing installation: tf_keras 2.19.0
Uninstalling tf_keras-2.19.0:
  Successfully uninstalled tf_keras-2.19.0
Found existing installation: keras 3.10.0
Uninstalling keras-3.10.0:
  Successfully uninstalled keras-3.10.0
Found existing installation: jax 0.7.2
Uninstalling jax-0.7.2:
  Successfully uninstalled jax-0.7.2
Found existing installation: jaxlib 0.7.2
Uninstalling jaxlib-0.7.2:
  Successfully uninstalled jaxlib-0.7.2
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 247.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 197.3 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 186.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
# Load quantized base model and tokenizer.
model_name = "Qwen/Qwen2.5-3B"

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

num_gpus = torch.cuda.device_count()
print(f"CUDA devices visible: {num_gpus}")
for i in range(num_gpus):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

if num_gpus < 1:
    raise RuntimeError("No CUDA GPU is visible in this runtime.")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

offload_dir = f"{WORK_DIR}/hf-offload"
os.makedirs(offload_dir, exist_ok=True)

max_memory = {i: "13GiB" for i in range(num_gpus)}
max_memory["cpu"] = "96GiB"

load_kwargs = dict(
    pretrained_model_name_or_path=model_name,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory=max_memory,
    offload_folder=offload_dir,
    offload_state_dict=True,
    low_cpu_mem_usage=True,
)

try:
    model = AutoModelForCausalLM.from_pretrained(**load_kwargs)
except torch.OutOfMemoryError:
    print("OOM on first load attempt. Retrying with tighter GPU caps...")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    tighter_max_memory = {i: "10GiB" for i in range(num_gpus)}
    tighter_max_memory["cpu"] = "120GiB"
    load_kwargs["max_memory"] = tighter_max_memory
    model = AutoModelForCausalLM.from_pretrained(**load_kwargs)

model.config.use_cache = False

# Default is base tokenizer for compatibility with pretrained embeddings.
if USE_CUSTOM_TOKENIZER and os.path.exists(CUSTOM_TOKENIZER_PATH):
    print(f"Loading custom tokenizer from: {CUSTOM_TOKENIZER_PATH}")
    tokenizer = PreTrainedTokenizerFast(
        tokenizer_file=CUSTOM_TOKENIZER_PATH,
        bos_token="<s>",
        eos_token="</s>",
        unk_token="<unk>",
        pad_token="<pad>",
    )
    model.resize_token_embeddings(len(tokenizer))
    print("Warning: embedding resize was applied for custom tokenizer; this is experimental and may reduce quality until further training.")
else:
    if USE_CUSTOM_TOKENIZER:
        print("Custom tokenizer path not found; falling back to base tokenizer.")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

print("Device map:", getattr(model, "hf_device_map", None))
print(f"Model loaded: {model.get_memory_footprint() / 1e9:.1f} GB")
print(f"Tokenizer vocab size: {len(tokenizer)}")

CUDA devices visible: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading custom tokenizer from: /kaggle/input/datasets/darksunnp/hamroai-cpt-v1/nepali_bpe_32k.json
Device map: {'model.embed_tokens': 0, 'lm_head': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.layers.32': 1, 'model.layers.33': 1, 'model.layers.34': 1, 'model.layers.35': 1, 'model.norm': 1, 'model.rotary_emb': 1}
Model loaded: 1.5 GB
Tokeni

In [6]:
# Attach LoRA adapters for trainable fine-tuning over quantized base.
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 59,867,136 || all params: 2,900,176,896 || trainable%: 2.0643


In [8]:
# Load corpus, split train/val, and build cached packed token blocks.
if not os.path.exists(CORPUS_PATH):
    raise FileNotFoundError(
        f"Corpus file not found at {CORPUS_PATH}. Set HAMROAI_CORPUS_PATH or upload the file there."
    )

dataset = load_dataset(
    "json",
    data_files=CORPUS_PATH,
    split="train",
)

print(f"Documents: {len(dataset):,}")
print(f"Sample: {dataset[0]['text'][:200]}")

split_ds = dataset.train_test_split(test_size=0.01, seed=42, shuffle=True)
train_ds = split_ds["train"]
val_ds = split_ds["test"]

if TRAIN_DOC_LIMIT > 0 and len(train_ds) > TRAIN_DOC_LIMIT:
    train_ds = train_ds.select(range(TRAIN_DOC_LIMIT))
if VAL_DOC_LIMIT > 0 and len(val_ds) > VAL_DOC_LIMIT:
    val_ds = val_ds.select(range(VAL_DOC_LIMIT))

print(f"Train docs (limited): {len(train_ds):,}")
print(f"Validation docs (limited): {len(val_ds):,}")

tokenizer_tag = "custom" if USE_CUSTOM_TOKENIZER else "base"
cache_key = f"qwen25_3b_{tokenizer_tag}_ml{MAX_LENGTH}_td{len(train_ds)}_vd{len(val_ds)}"
train_cache_path = f"{TOKENIZED_CACHE_DIR}/{cache_key}/train"
val_cache_path = f"{TOKENIZED_CACHE_DIR}/{cache_key}/val"

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=False, padding=False)

def pack_to_blocks(examples):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])
    total_length = (total_length // MAX_LENGTH) * MAX_LENGTH
    result = {
        k: [t[i : i + MAX_LENGTH] for i in range(0, total_length, MAX_LENGTH)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

if os.path.exists(train_cache_path) and os.path.exists(val_cache_path):
    print("Loading tokenized datasets from cache...")
    tokenized_train = load_from_disk(train_cache_path)
    tokenized_val = load_from_disk(val_cache_path)
else:
    print("Tokenizing and packing datasets, then saving cache...")
    tok_train = train_ds.map(
        tokenize_fn,
        batched=True,
        remove_columns=train_ds.column_names,
        num_proc=NUM_PROC,
        desc="Tokenizing train split",
    )
    tok_val = val_ds.map(
        tokenize_fn,
        batched=True,
        remove_columns=val_ds.column_names,
        num_proc=NUM_PROC,
        desc="Tokenizing validation split",
    )

    tokenized_train = tok_train.map(
        pack_to_blocks,
        batched=True,
        batch_size=1000,
        num_proc=NUM_PROC,
        desc="Packing train into fixed blocks",
    )
    tokenized_val = tok_val.map(
        pack_to_blocks,
        batched=True,
        batch_size=1000,
        num_proc=NUM_PROC,
        desc="Packing val into fixed blocks",
    )

    os.makedirs(os.path.dirname(train_cache_path), exist_ok=True)
    tokenized_train.save_to_disk(train_cache_path)
    tokenized_val.save_to_disk(val_cache_path)
    print("Tokenized cache saved.")

print(f"Tokenized train examples: {len(tokenized_train):,}")
print(f"Tokenized val examples: {len(tokenized_val):,}")

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/50 [00:00<?, ?it/s]

Documents: 6,803,611
Sample: २०८१ साउन ८ मंगलबार पौष २१ काठमाडौं कतिपय जनप्रतिनिधिहरू विकास निर्माण कार्यको ठेकेदारीमा आफैँ संलग्न भएको भन्दै आलोचना भइरहेका बेला बाराको कोहल्वी नगरपालिकामा पनि सोही प्रवृत्ति देखिएको छ । विगतमा ठे
Train docs (limited): 300,000
Validation docs (limited): 4,000
Tokenizing and packing datasets, then saving cache...


Tokenizing train split (num_proc=4):   0%|          | 0/300000 [00:00<?, ? examples/s]

Tokenizing validation split (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

Packing train into fixed blocks (num_proc=4):   0%|          | 0/300000 [00:00<?, ? examples/s]

Packing val into fixed blocks (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

Saving the dataset (0/3 shards):   0%|          | 0/145505 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1896 [00:00<?, ? examples/s]

Tokenized cache saved.
Tokenized train examples: 145,505
Tokenized val examples: 1,896


In [7]:
# Single-process trainer setup (skip this cell when running DDP launch cell).
os.environ["USE_TORCH"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"

if os.environ.get("HAMROAI_USE_DDP", "0") == "1":
    print("HAMROAI_USE_DDP=1 -> skipping single-process Trainer setup cell.")
else:
    if not isinstance(model, PeftModel):
        print("Model is not PEFT-wrapped. Attaching LoRA adapters now...")
        model = prepare_model_for_kbit_training(model)
        lora_config = LoraConfig(
            r=32,
            lora_alpha=64,
            target_modules=[
                "q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj",
            ],
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
        )
        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()
    else:
        print("Model already has PEFT adapters attached.")

    if "tokenized_train" not in globals() or "tokenized_val" not in globals():
        print("tokenized_* not found in memory. Attempting to load from cache...")
        loaded = False

        if "train_cache_path" in globals() and "val_cache_path" in globals():
            if os.path.exists(train_cache_path) and os.path.exists(val_cache_path):
                tokenized_train = load_from_disk(train_cache_path)
                tokenized_val = load_from_disk(val_cache_path)
                loaded = True

        if not loaded and "TOKENIZED_CACHE_DIR" in globals() and os.path.isdir(TOKENIZED_CACHE_DIR):
            root = Path(TOKENIZED_CACHE_DIR)
            candidates = [p for p in root.iterdir() if (p / "train").exists() and (p / "val").exists()]
            if candidates:
                latest = max(candidates, key=lambda p: p.stat().st_mtime)
                tokenized_train = load_from_disk(str(latest / "train"))
                tokenized_val = load_from_disk(str(latest / "val"))
                loaded = True
                print(f"Loaded cache set: {latest.name}")

        if not loaded:
            raise RuntimeError(
                "tokenized_train/tokenized_val not found. Run the tokenization cell once to create cache, then rerun this cell."
            )

    print(f"Trainer using train examples: {len(tokenized_train):,}")
    print(f"Trainer using val examples: {len(tokenized_val):,}")

    training_args = TrainingArguments(
        output_dir=CHECKPOINT_DIR,
        max_steps=MAX_STEPS,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-5,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        fp16=True,
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
        save_strategy="steps",
        save_steps=50,
        save_total_limit=3,
        eval_strategy="no",
        logging_steps=50,
        report_to="none",
        dataloader_num_workers=0,
     )

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
     )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        data_collator=data_collator,
     )

Model already has PEFT adapters attached.
tokenized_* not found in memory. Attempting to load from cache...


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs


Loaded cache set: qwen25_3b_custom_ml512_td300000_vd4000
Trainer using train examples: 145,505
Trainer using val examples: 1,896


In [ ]:
# Deprecated duplicate trainer-setup cell.
# Use the previous trainer setup cell only.
# Keeping this cell empty prevents accidental NameError from stale kernel state.

**ONLY RUN TO DELETE THE CHECKPOINT DIRECTORY**

In [10]:
!rm -rf /kaggle/working/hamroai-cpt-checkpoints

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


**CAUTION**

In [8]:
# Train/resume for single-process mode.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

resume_ckpt = None
if os.path.isdir(CHECKPOINT_DIR):
    resume_ckpt = trainer_utils.get_last_checkpoint(CHECKPOINT_DIR)

if resume_ckpt:
    print(f"Resuming from checkpoint: {resume_ckpt}")
    trainer.train(resume_from_checkpoint=resume_ckpt)
else:
    print("No checkpoint found. Starting fresh training run.")
    trainer.train()

Resuming from checkpoint: /kaggle/working/hamroai-cpt-checkpoints/checkpoint-1100


/usr/local/lib/python3.12/dist-packages/transformers/trainer.py:3262: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(checkpoint, OPTIMIZER_NAME), map_

Step,Training Loss
1150,7.436900
1200,7.446200
1250,7.451500
1300,7.440100
1350,7.373200
1400,7.410400
1450,7.363700
1500,7.362900
1550,7.444200
1600,7.342000


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:232: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:232: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:232: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:232: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:232: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetunin

In [ ]:
# Optional manual resume cell (only if you want to force a specific checkpoint path).
# trainer.train(resume_from_checkpoint="/path/to/checkpoint")

In [ ]:
# Save LoRA adapter artifacts.
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Saved adapter to: {ADAPTER_DIR}")

#set push_to_hub = os.environ.get("HAMROAI_PUSH_TO_HUB", "1") == "1" to push to hf
push_to_hub = os.environ.get("HAMROAI_PUSH_TO_HUB", "0") == "1"
if push_to_hub and os.environ.get("HF_TOKEN"):
    model.push_to_hub("darksunnp/hamroai-nepali-lora-v1")
    tokenizer.push_to_hub("darksunnp/hamroai-nepali-lora-v1")
    print("Pushed adapter to Hugging Face Hub.")
else:
    print("Skipping push_to_hub (set HAMROAI_PUSH_TO_HUB=1 and HF_TOKEN to enable).")

Saved adapter to: /kaggle/working/hamroai-nepali-lora
Skipping push_to_hub (set HAMROAI_PUSH_TO_HUB=1 and HF_TOKEN to enable).


In [8]:
# Test generation
prompt = "नेपालको राजधानी"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.7)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


नेपालको राजधानी भएको छ । नेपालको राजधानी भएको छ । नेपालको राजधानी भएको छ । नेपालको राजधानी भएको छ । नेपालको राजधानी


In [ ]:
# Create a 2-GPU DDP training script for better GPU utilization balance.
from pathlib import Path

script_path = Path(WORK_DIR) / "train_ddp_lora.py"
script = r'''
import argparse
import gc
import os
from pathlib import Path

os.environ["USE_TORCH"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"

import torch
from datasets import load_from_disk
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
    trainer_utils,
    PreTrainedTokenizerFast,
  )

def find_latest_cache_root(root_dir: str) -> Path:
    root = Path(root_dir)
    candidates = [p for p in root.iterdir() if (p / "train").exists() and (p / "val").exists()]
    if not candidates:
        raise RuntimeError(f"No cached train/val pair found under {root_dir}")
    return max(candidates, key=lambda p: p.stat().st_mtime)

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--model_name", default="Qwen/Qwen2.5-3B")
    p.add_argument("--checkpoint_dir", required=True)
    p.add_argument("--cache_root", required=True)
    p.add_argument("--train_cache", default="")
    p.add_argument("--val_cache", default="")
    p.add_argument("--max_steps", type=int, default=2200)
    p.add_argument("--lr", type=float, default=2e-5)
    p.add_argument("--grad_accum", type=int, default=8)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--use_custom_tokenizer", action="store_true")
    p.add_argument("--custom_tokenizer_path", default="")
    return p.parse_args()

def main():
    args = parse_args()
    set_seed(args.seed)

    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    world_size = int(os.environ.get("WORLD_SIZE", "1"))
    device = f"cuda:{local_rank}"
    torch.cuda.set_device(local_rank)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        quantization_config=bnb_config,
        device_map={"": local_rank},
        low_cpu_mem_usage=True,
    )
    model.config.use_cache = False

    if args.use_custom_tokenizer and args.custom_tokenizer_path and os.path.exists(args.custom_tokenizer_path):
        tokenizer = PreTrainedTokenizerFast(
            tokenizer_file=args.custom_tokenizer_path,
            bos_token="<s>",
            eos_token="</s>",
            unk_token="<unk>",
            pad_token="<pad>",
        )
        model.resize_token_embeddings(len(tokenizer))
    else:
        tokenizer = AutoTokenizer.from_pretrained(args.model_name)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

    model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(
        r=32,
        lora_alpha=64,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)

    if args.train_cache and args.val_cache and os.path.exists(args.train_cache) and os.path.exists(args.val_cache):
        train_cache = Path(args.train_cache)
        val_cache = Path(args.val_cache)
    else:
        latest = find_latest_cache_root(args.cache_root)
        train_cache = latest / "train"
        val_cache = latest / "val"
        if local_rank == 0:
            print(f"Using latest cache set: {latest}")

    tokenized_train = load_from_disk(str(train_cache))
    tokenized_val = load_from_disk(str(val_cache))

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    training_args = TrainingArguments(
        output_dir=args.checkpoint_dir,
        max_steps=args.max_steps,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.lr,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        fp16=True,
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
        save_strategy="steps",
        save_steps=100,
        save_total_limit=3,
        eval_strategy="no",
        logging_steps=50,
        report_to="none",
        dataloader_num_workers=2,
        ddp_find_unused_parameters=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        data_collator=data_collator,
    )

    resume_ckpt = None
    if os.path.isdir(args.checkpoint_dir):
        resume_ckpt = trainer_utils.get_last_checkpoint(args.checkpoint_dir)

    if local_rank == 0:
        print(f"WORLD_SIZE={world_size}, LOCAL_RANK={local_rank}, DEVICE={device}")
        print(f"Train examples: {len(tokenized_train):,}; Val examples: {len(tokenized_val):,}")
        if resume_ckpt:
            print(f"Resuming from checkpoint: {resume_ckpt}")

    if resume_ckpt:
        trainer.train(resume_from_checkpoint=resume_ckpt)
    else:
        trainer.train()

    if local_rank == 0:
        out_dir = Path(args.checkpoint_dir) / "final_adapter"
        out_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(str(out_dir))
        tokenizer.save_pretrained(str(out_dir))
        print(f"Saved final adapter to: {out_dir}")

    gc.collect()
    torch.cuda.empty_cache()

if __name__ == "__main__":
    main()
'''
script_path.write_text(script, encoding="utf-8")
print(f"Wrote DDP training script: {script_path}")

Wrote DDP training script: /kaggle/working/train_ddp_lora.py


In [ ]:
# Launch true 2-GPU data parallel training via torchrun.
for name in ["trainer", "model", "tokenized_train", "tokenized_val"]:
    if name in globals():
        del globals()[name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

train_cache = ""
val_cache = ""
if "train_cache_path" in globals() and "val_cache_path" in globals():
    if os.path.exists(train_cache_path) and os.path.exists(val_cache_path):
        train_cache = train_cache_path
        val_cache = val_cache_path

env = os.environ.copy()
env["USE_TORCH"] = "1"
env["USE_TF"] = "0"
env["USE_FLAX"] = "0"
env["TRANSFORMERS_NO_TF"] = "1"
env["TRANSFORMERS_NO_FLAX"] = "1"

cmd = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",
    f"{WORK_DIR}/train_ddp_lora.py",
    "--model_name", "Qwen/Qwen2.5-3B",
    "--checkpoint_dir", CHECKPOINT_DIR,
    "--cache_root", TOKENIZED_CACHE_DIR,
    "--max_steps", str(MAX_STEPS),
    "--grad_accum", "8",
]

if train_cache and val_cache:
    cmd.extend(["--train_cache", train_cache, "--val_cache", val_cache])

if USE_CUSTOM_TOKENIZER and os.path.exists(CUSTOM_TOKENIZER_PATH):
    cmd.extend(["--use_custom_tokenizer", "--custom_tokenizer_path", CUSTOM_TOKENIZER_PATH])

print("Running:", " ".join(cmd))
subprocess.run(cmd, env=env, check=True)

Running: torchrun --standalone --nproc_per_node=2 /kaggle/working/train_ddp_lora.py --model_name Qwen/Qwen2.5-3B --checkpoint_dir /kaggle/working/hamroai-cpt-checkpoints --cache_root /kaggle/working/hamroai-tokenized-cache --max_steps 2200 --grad_accum 8 --train_cache /kaggle/working/hamroai-tokenized-cache/qwen25_3b_base_ml512_td150000_vd2000/train --val_cache /kaggle/working/hamroai-tokenized-cache/qwen25_3b_base_ml512_td150000_vd2000/val


W0317 13:26:09.197000 3011 torch/distributed/run.py:793] 
W0317 13:26:09.197000 3011 torch/distributed/run.py:793] *****************************************
W0317 13:26:09.197000 3011 torch/distributed/run.py:793] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0317 13:26:09.197000 3011 torch/distributed/run.py:793] *****************************************
Loading checkpoint shards: 100%|██████████| 2/2 [00:11<00:00,  5.88s/it]
[W317 13:26:31.005466312 CUDAAllocatorConfig.h:28] Warning: expandable_segments not supported on this platform (function operator())
[W317 13:26:31.034311116 CUDAAllocatorConfig.h:28] Warning: expandable_segments not supported on this platform (function operator())
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. 

WORLD_SIZE=2, LOCAL_RANK=0, DEVICE=cuda:0
Train examples: 376,027; Val examples: 4,776


  2%|▏         | 50/2200 [09:51<7:03:22, 11.82s/it]

{'loss': 1.718, 'grad_norm': 0.48406165838241577, 'learning_rate': 9.090909090909091e-06, 'epoch': 0.0}


...

{'loss': 1.1377, 'grad_norm': 1.4752181768417358, 'learning_rate': 1.3032931040502627e-05, 'epoch': 0.04}


 45%|████▌     | 1000/2200 [3:16:58<3:55:40, 11.78s/it]

{'loss': 1.1284, 'grad_norm': 1.592116117477417, 'learning_rate': 1.2308867204447958e-05, 'epoch': 0.04}


 47%|████▋     | 1034/2200 [3:23:41<3:49:14, 11.80s/it]

: 